# D: continuous angle commands / непрерывное задание угла

The input is the desired angle in degrees. The current polynomials, neurons, muscles, full afferents and limb evolve continuously. No PD/PID controller or hidden-state reset is used between commands. See [README](README.md) / [English](README.en.md). Training is never launched by this notebook.

Вход — желаемый угол, фактическое движение может отставать. Все графики ниже вычисляются из сохранённого генома; начальные состояния и параметры сигнала можно менять.

<!-- current-training -->
**Переобучение: времена мышц ≥50 мс.** Геном `C_transfer`. Завершено тестов 14/18, строгий успех 0/18. Перезапустите ядро. [Результаты](results/README.md).

**English.** Muscle times ≥50 ms; winner `C_transfer`. Completed 14/18; strict success 0/18. Restart the kernel. [Results](results/README.en.md).


**Проверка этой серии:** резкий переход цели +45° → −45° вызвал падение через 0,5864 с. После остановки используйте «Сброс».

**English:** the +45° → −45° target step caused a fall after 0.5864 s in this study; use Reset after the simulation stops.


<!-- mirror-genome-variants -->

Три сохранённые версии: [быстрые мышцы](best_genome_fast_muscle.json), [времена ≥50 мс](best_genome_slow_muscle.json), [времена ≥50 мс и симметрия](best_genome_slow_muscle_mirrow.json). Выберите версию через `GENOME_VARIANT`; `GENOME_PATH` позволяет указать другой файл. По умолчанию выбран зеркальный геном. Повторный запуск ячейки обновляет старые загрузчики в памяти ядра. [Происхождение и параметры](GENOME_VERSIONS.md).

Three saved variants: fast muscles, ≥50 ms muscles, and ≥50 ms with exact mirror symmetry. Select a named version using `GENOME_VARIANT`, or override it with `GENOME_PATH`. The mirror genome is selected by default. Rerunning the cell refreshes stale cached codecs. [Sources and values](GENOME_VERSIONS.en.md).

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/SpikingNN').is_dir())
sys.path.insert(0, str(REPO))
from scripts.AfferentBalance2026.experiments.core.config import project_root
from scripts.AfferentBalance2026.experiments.notebook_runtime import select_solution, controller_input
from scripts.AfferentBalance2026.experiments.visualization import (
    configure_fonts, replay_grid, parameter_scan, plot_phase, plot_angles, plot_states,
)
from scripts.AfferentBalance2026.experiments.simulation import case_grid, simulate
PROJECT = project_root()
SMOKE_TEST = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') == '1'
SMOKE = SMOKE_TEST
EXPERIMENT = PROJECT/'experiments/D_trajectory'
FOLDER = EXPERIMENT
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None  # Explicit JSON/NPZ/pointer overrides the selected variant.
USE_INITIAL_CONTROLLER = False  # Explicit untrained configuration; smoke never reads ignored results.
RANK = 1
solution = select_solution(EXPERIMENT, GENOME_PATH, variant=GENOME_VARIANT, rank=RANK,
                           smoke=SMOKE_TEST, initial=USE_INITIAL_CONTROLLER)
config = simulation_config = cfg = solution.config
physical = solution.physical.copy()
selection = metadata = solution.metadata
GENOME_PATH = controller_input(solution)
recording_stride = 10  # Plot recording only; fitness still uses every 0.1 ms step.
TARGET_DEG = 0.0
CURRENTS = np.array([0.0, 0.0])
configure_fonts(13)
from scripts.AfferentBalance2026.experiments.D_trajectory.controller import AngleController, TrajectoryGenome
from scripts.AfferentBalance2026.experiments.D_trajectory.trajectory import replay, sine_case, command_currents
coefficients, tau_s = solution.command, solution.current_tau_s
protocol = {'kind': 'sine', 'amplitude_deg': 5, 'period_s': 50}


## Stateful input / подача угла
`advance(angle, duration_s)` holds the command for a block with a 0.1 ms internal step. Assigning `target_angle_deg` changes only the command. `reset` is explicit.

In [ ]:
controller = AngleController(GENOME_PATH, initial_angle_deg=-30, target_angle_deg=0)
state = controller.advance(15, duration_s=0.01)
controller.target_angle_deg = 10
state = controller.step()
print({key: state[key] for key in ("time_s", "target_angle_deg", "angle_deg", "input_currents")})

In [ ]:
def run_sine(amplitude_deg=5, period_s=50, initial_angle_deg=0, initial_speed_rad_s=0,
             p=None, c=None):
    """Replay one continuous sine command; retain every 0.1 ms sample."""
    spec = sine_case(amplitude_deg, period_s, initial_angle=initial_angle_deg,
                     initial_speed=initial_speed_rad_s)
    if SMOKE:
        spec.update(duration_s=0.02, hold_s=0.002, ramp_s=0.002)
    result = replay(physical if p is None else p, coefficients if c is None else c,
                    cfg.mechanics, spec, tau_s=tau_s)
    return spec, result


def view_states(result):
    """Show angle, currents, neurons, muscles, afferents and the actual phase trajectory."""
    tr, ref, currents = result["trace"], result["reference"], result["commands"]
    sample = np.unique(np.r_[np.arange(0, len(tr), 1 if SMOKE else 50), len(tr)-1])
    t = tr[sample, 0]
    fig, axes = plt.subplots(4, 2, figsize=(15, 17), layout="constrained")
    axes[0, 0].plot(t, np.rad2deg(tr[sample, 1]), label="Limb")
    axes[0, 0].plot(t, np.rad2deg(ref[sample, 0]), "k--", label="Target")
    axes[0, 1].plot(np.rad2deg(tr[sample, 1]), tr[sample, 2], label="Limb")
    axes[0, 1].plot(np.rad2deg(ref[sample, 0]), ref[sample, 1], "k--", label="Target")
    axes[1, 0].plot(t, currents[sample, :2], label=["Neuron 0", "Neuron 1"])
    window = np.arange(len(tr)) < 3000
    axes[1, 1].plot(tr[sample, 0], tr[sample, 5:7], label=["Neuron 0", "Neuron 1"], lw=.8)
    axes[2, 0].plot(t, tr[sample, 3:5], label=["Flexor", "Extensor"])
    axes[2, 1].plot(t, tr[sample, 17:19], label=["Flexor", "Extensor"])
    axes[3, 0].plot(t, tr[sample, 11:14], label=["Ia", "II", "Ib"])
    axes[3, 1].plot(t, tr[sample, 14:17], label=["Ia", "II", "Ib"])
    labels = [("Limb", "Angle (deg)"), ("Phase trajectory", "Velocity (rad/s)"),
              ("Applied currents", "Current (model units)"), ("Membrane potentials", "mV"),
              ("Muscle forces", "N"), ("Hill activation", "Activation"),
              ("Flexor afferents", "Response"), ("Extensor afferents", "Response")]
    for i, (ax, (title, ylabel)) in enumerate(zip(axes.ravel(), labels)):
        ax.set(title=title, xlabel="Angle (deg)" if i == 1 else "Time (s)", ylabel=ylabel)
        ax.legend()
        ax.grid(alpha=.25)
    return fig

spec, result = run_sine()
print("Tracking RMSE (deg):", np.rad2deg(result["metrics"][2]), "failure:", result["failure"])
fig = view_states(result)

In [ ]:
def phase_grid(angles_deg=(-45, 0, 45), speeds_rad_s=(-0.5, 0.5), amplitude_deg=30, period_s=20):
    """Simulate distinct initial conditions under the same moving angle command."""
    fig, ax = plt.subplots(figsize=(10, 7))
    rows = []
    for theta in angles_deg:
        for omega in speeds_rad_s:
            _, out = run_sine(amplitude_deg, period_s, theta, omega)
            full_trace = out["trace"]
            indices = np.unique(np.r_[np.arange(0, len(full_trace), 1 if SMOKE else 100), len(full_trace)-1])
            tr = full_trace[indices]
            ax.plot(np.rad2deg(tr[:, 1]), tr[:, 2], lw=.8, label=f"{theta:g}°, {omega:g} rad/s")
            if out["failure"]:
                ax.plot(np.rad2deg(tr[-1, 1]), tr[-1, 2], "kx", ms=7)
            rows.append((theta, omega, np.rad2deg(out["metrics"][2]), out["failure"]))
    ax.set(xlabel="Angle (deg)", ylabel="Angular velocity (rad/s)", title="Continuous tracking from different initial states")
    ax.legend(fontsize=11)
    ax.grid(alpha=.25)
    return fig, rows

# For a 10-by-10 grid: phase_grid(np.linspace(-50, 50, 10), np.linspace(-1, 1, 10)).
fig, grid_metrics = phase_grid(angles_deg=(0,) if SMOKE else (-45, 0, 45), speeds_rad_s=(0,) if SMOKE else (-.5, .5))
grid_metrics

In [ ]:
def scan_parameter(name, values):
    """Scan a physical parameter, preserving fixed values and optional mirror pairs."""
    from scripts.AfferentBalance2026.experiments.core.genome import MIRROR_INDICES
    index = next(i for i, field in enumerate(cfg.data["parameters"]) if field["name"] == name)
    if index not in cfg.mutable:
        raise ValueError("This parameter is fixed by the genome configuration")
    rows = []
    for value in values:
        p = physical.copy()
        p[index] = value
        if cfg.mirror_symmetry:
            p[MIRROR_INDICES[index]] = value  # Preserve the reflected physical pair.
        cfg.encode(p)
        _, out = run_sine(p=p)
        rows.append((value, np.rad2deg(out["metrics"][2]), out["failure"]))
    return np.asarray(rows)

# Example: scan_parameter("muscles.flexor.tau_c", [50, 71, 100]).
# Input-current curve already encoded in this genome:
targets = np.linspace(-45, 45, 181)
commands = np.array([command_currents(coefficients, np.deg2rad(angle)) for angle in targets])
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(targets, commands, label=["Neuron 0", "Neuron 1"])
ax.set(xlabel="Desired angle (deg)", ylabel="Current (model units)", title="Genome current law")
ax.legend()
ax.grid(alpha=.25)

In [ ]:
# LIVE_ANGLE_SIMULATION: choose a solution and settings; numerical state is never auto-reloaded.
from pathlib import Path
import os
import sys
from IPython.display import display
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/SpikingNN').is_dir())
sys.path.insert(0, str(REPO))
from scripts.AfferentBalance2026.experiments.core.config import project_root
from scripts.AfferentBalance2026.experiments.notebook_runtime import select_solution, controller_input
from scripts.AfferentBalance2026.experiments.notebook_ui import LivePendulumActivitySimulation, DEFAULT_SETTINGS

# ---------------- USER SETTINGS: change these and rerun this cell ----------------
# Choose fast_muscle, slow_muscle, slow_muscle_mirrow, or current.
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None              # Optional explicit path; overrides GENOME_VARIANT.
INITIAL_ANGLE_DEG = 0.0         # Actual initial angle relative to the unstable vertical.
INITIAL_SPEED_RAD_S = 0.0       # Actual initial angular velocity.
INITIAL_TARGET_DEG = 0.0        # Slider starts at this desired angle, within [-45, 45].
TARGET_STEP_DEG = 0.5           # Slider resolution; currents are evaluated by the saved polynomial.
WINDOW_S = 10.0                 # Duration of the rolling angle plot, in simulated seconds.
COMMAND_INTERVAL_S = 0.01       # Command/sample interval; physical integration is still 0.0001 s.
FPS = 20                       # Maximum display refresh rate, independent of the physics step.
TIME_SCALE = 1.0                # 1.0 = one simulated second per wall-clock second.
MAX_BLOCKS_PER_FRAME = 10       # Limit catch-up work so slider/buttons remain responsive.
AUTO_START = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') != '1'  # Set False to start manually.

FONT_FAMILY = 'Times New Roman' # Font used for all labels inside the figure.
FONT_SIZE = 15                  # Base figure font size.
TITLE_SIZE = 15                 # Panel titles.
LABEL_SIZE = 15                 # Axis labels.
TICK_SIZE = 15                  # Tick labels.
LEGEND_SIZE = 15                # Legend text.
ANNOTATION_SIZE = 11            # Angles, time and force values printed on the diagram.
WIDGET_FONT_SIZE = 15           # Text size of the HTML status/readout, in pixels.
FIGSIZE = (12.5, 5.5)           # Figure size in inches: schematic on the left, traces on the right.
FRAME_DPI = 95                 # Interactive image resolution; lower this if display is slow.
SNAPSHOT_DPI = 220              # Resolution of the PNG saved by the snapshot button.
FORCE_SCALE_M_PER_N = 0.004     # Base display length per newton, before visual length limits.
GRAVITY_MIN_ARROW_M = 0.06      # Minimum visible gravity-arrow length; does not change physical mg.
MAX_ARROW_M = 0.16              # Cap display length only; printed force values remain unchanged.
SNAPSHOT_DIR = None             # None saves under D/results/live_snapshots; no helper files are needed.
FLEXOR_COLOR = '#d62728'         # Flexor muscle: red.
EXTENSOR_COLOR = '#1764c0'       # Extensor muscle: blue.
TARGET_COLOR = '#e6272f'         # Desired direction and desired-angle curve: green.
LIMB_COLOR = '#000000'           # Actual limb and actual-angle curve: dark gray.


SMOKE = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') == '1'
EXPERIMENT = project_root()/'experiments/D_trajectory'
solution = select_solution(EXPERIMENT, GENOME_PATH, variant=GENOME_VARIANT, smoke=SMOKE)
protocol = {'initial_angle_deg': INITIAL_ANGLE_DEG, 'initial_speed_rad_s': INITIAL_SPEED_RAD_S}
recording_stride = 1  # Activity records every physical 0.1 ms step.
settings = {name: globals()[name] for name in DEFAULT_SETTINGS if name in globals()}
_previous = globals().get('live_pendulum')
if _previous is not None:
    _previous.close()
live_pendulum = LivePendulumActivitySimulation(
    controller_input(solution), SNAPSHOT_DIR or EXPERIMENT/'results/live_snapshots',
    settings=settings, activity=False)
display(live_pendulum.ui)
if AUTO_START and not SMOKE:
    live_pendulum.start()
# Explicit controls: .stop(), .start(), .save_snapshot(); no training is performed.


<!-- local-refactor-20261027 -->
**Загрузка:** все виды просмотра используют `solution`, явный `protocol` и `recording_stride`.
При изменении вычислительного кода перезапустите ядро. Старые сохранённые outputs ниже оставлены
как история и не пересчитывались при рефакторинге. `SPIKINGNN_NOTEBOOK_SMOKE=1` использует
необученный геном в памяти, короткий горизонт и не запускает GA.

**English:** Views share one selected solution and explicit protocol/recording stride. Restart the
kernel after numerical source changes. Existing outputs are preserved historical observations.
Smoke mode uses an untrained in-memory fixture and never trains.
